# Lesson 5: Building the Agent

We have an MCP server with 14 tools (Lesson 4) and a clean data API behind it (Lesson 3). Today we build the agent that *uses* those tools to answer real questions — and we add a human approval step before it can write anything back to the system.

## The architecture so far

```
Lesson 2:  SQLite          →  OpenSearch index (machine_usage)
                                       ↓
Lesson 3:  FastAPI         ←  http://localhost:8000  (12 endpoints)
                                       ↓
Lesson 4:  MCP server      ←  http://localhost:8001  (14 tools)
                                       ↓
Lesson 5:  Agent           ←  this notebook
```

**Prerequisite:** The MCP server from Lesson 4 must be running on port 8001, and `OPENAI_API_KEY` must be set in `.env` at the repo root.

## 1. What is an agent?

An agent is three things working in a loop:

1. **A model** (the LLM) — decides what to do next.
2. **A set of tools** — the things it's allowed to do (our MCP server).
3. **A loop** — call the model, run any tool calls it asks for, feed results back, repeat until the model is done.

That's it. Everything else (instructions, memory, handoffs, approvals) is layered on top.

Contrast with a plain LLM call: a chat completion returns one response and stops. An agent can take five tool-call hops to answer a single question — and that's where the value is, because real questions rarely match a single endpoint.

### Where the layers separate responsibility

| Layer | Owns | Doesn't own |
|---|---|---|
| **API** (Lesson 3) | Data access, query shape, validation | Who's allowed to call what |
| **MCP tools** (Lesson 4) | Names, descriptions, parameter contracts for the LLM | The conversation, the goal |
| **Agent** (today) | Goal, instructions, conversation, approval policy | Data shape, tool implementation |

Clean separation means each layer can be tested, replaced, and reasoned about independently.

---
## 2. Setup

Load the OpenAI key and confirm the MCP server from Lesson 4 is reachable.

In [ ]:
import os
import json
import socket
from pathlib import Path
from dotenv import load_dotenv

_env_path = Path().resolve().parent / '.env'
if not _env_path.exists():
    _env_path = Path().resolve() / '.env'
load_dotenv(_env_path)

assert os.environ.get('OPENAI_API_KEY'), (
    f'OPENAI_API_KEY not found. Create a .env file at {_env_path.parent} with OPENAI_API_KEY=sk-...'
)
print('OPENAI_API_KEY loaded ✓')

In [ ]:
_check = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
_check.settimeout(1.0)
_mcp_up = _check.connect_ex(('localhost', 8001)) == 0
_check.close()

assert _mcp_up, (
    'MCP server is not reachable on port 8001. '
    'Run mcp_tools/lesson_04_mcp_tools.ipynb first and leave it running.'
)
print('MCP server reachable on http://localhost:8001 ✓')

---
## 3. Connect to the MCP server

The OpenAI Agents SDK ships an `MCPServerSse` client that handles the protocol for us. We pass it the URL, it negotiates the connection and discovers the tools.

This is the contract handoff: Lesson 4 published the tools, Lesson 5 consumes them. The MCP layer is the seam.

In [ ]:
from agents.mcp import MCPServerSse, MCPServerSseParams

MCP_URL = 'http://localhost:8001/sse'

async with MCPServerSse(params=MCPServerSseParams(url=MCP_URL)) as mcp_server:
    tools = await mcp_server.list_tools()
    print(f'Discovered {len(tools)} tools from {MCP_URL}:\n')
    for t in sorted(tools, key=lambda t: t.name):
        first_line = (t.description or '').strip().split('\n')[0]
        print(f'  • {t.name:42s}  {first_line}')

---
## 4. Build the agent

An agent in the OpenAI Agents SDK is just a dataclass: a name, a model, a system prompt, and the tools it can use.

The hard part isn't the code — it's the **instructions**. The instructions are how you shape behavior. Read them as English.

### Anatomy of good agent instructions

Five things every system prompt should answer:

1. **Role** — who is the agent? (`You are a manufacturing maintenance analyst...`)
2. **Scope** — what's in bounds, what's out?
3. **Tool guidance** — when to prefer which tool, and how to chain them. The tool descriptions already say this, but reinforcing the *strategy* helps.
4. **Stop condition** — when is the agent done? (Answer produced, evidence cited, etc.)
5. **Output style** — cite IDs, show numbers, no hand-waving.

Treat instructions like a product spec. Version them. If a behavior change is needed, edit the prompt before reaching for code.

In [ ]:
from agents import Agent

INSTRUCTIONS = (
    'You are a manufacturing maintenance analyst. '
    'Investigate machine usage and maintenance issues using the available tools.\n\n'
    'Tool strategy:\n'
    '- When the user names a failure type in plain English (e.g. "hydraulic pressure loss"), '
    'resolve it with find_reason_code_by_description BEFORE filtering.\n'
    '- Start broad (rank / compare), then drill into specifics (records / personnel).\n'
    '- When discussing personnel or maintenance reason codes, use the tools available to look up the semantic names, '
    'and discuss using text not IDs. IDs should only be used as parameters.\n'
    '- For training questions, prefer analyze_training_needs — it runs the full pipeline.\n\n'
    'Output:\n'
    '- Cite specific IDs and metrics from the data, never paraphrase from memory.\n'
    '- If a tool returns no results, say so plainly instead of inventing an answer.\n'
    '- Stop when you have evidence that answers the question.'
)

print(INSTRUCTIONS)

---
## 5. Run a read-only query

Let's ask a real question and watch the agent work. We'll stream the run so we can see every tool call as it happens — this is the **observability** layer, and it's non-negotiable for production.

If you can't see what the agent did, you can't trust the answer.

In [ ]:
def print_tool_event(event):
    """Render a streamed event from Runner.run_streamed as a readable log line."""
    if not hasattr(event, 'item'):
        return
    item = event.item
    kind = type(item).__name__

    if kind == 'ToolCallItem':
        raw = item.raw_item
        name = raw.get('name') if isinstance(raw, dict) else getattr(raw, 'name', '?')
        args_raw = raw.get('arguments', '{}') if isinstance(raw, dict) else getattr(raw, 'arguments', '{}')
        try:
            args = json.loads(args_raw) if isinstance(args_raw, str) else args_raw
        except Exception:
            args = args_raw
        print(f'\n🔧 {name}({json.dumps(args)})')

    elif kind == 'ToolCallOutputItem':
        out = item.output
        try:
            if isinstance(out, list):
                preview = [r.model_dump() if hasattr(r, 'model_dump') else r for r in out][:3]
                more = f' ... (+{len(out)-3} more)' if len(out) > 3 else ''
                print(f'   → {json.dumps(preview, default=str)}{more}')
            else:
                print(f'   → {out}')
        except Exception:
            print(f'   → {out}')

In [ ]:
from agents import Runner

QUERY = 'Which three machines have the most maintenance downtime, and what failure types dominate them?'

async with MCPServerSse(params=MCPServerSseParams(url=MCP_URL)) as mcp_server:
    agent = Agent(
        name='maintenance_analyst',
        model='gpt-4o-mini',
        instructions=INSTRUCTIONS,
        mcp_servers=[mcp_server],
    )

    print(f'Query: {QUERY}')
    print('═' * 70)
    result = Runner.run_streamed(agent, QUERY)
    async for event in result.stream_events():
        print_tool_event(event)

    print('\n' + '═' * 70)
    print('Final answer:\n')
    print(result.final_output)

### What you just saw

The agent picked tools, passed outputs forward, and stopped when it had enough to answer. We didn't prescribe the chain — the tool descriptions did.

In production this log is what you ship to your tracing system (OpenTelemetry, Langfuse, the OpenAI dashboard, etc.). Every tool call should be inspectable after the fact:
- What was asked?
- What tools were called, with what arguments?
- What did each tool return?
- What was the final answer?

Without this, the agent is a black box — and black boxes don't survive contact with real users.

---
## 6. Human-in-the-loop on the write tool

Reading data is one thing. Writing data — submitting a retraining request that a real person will receive — is another. We never want the agent to write without a human seeing the evidence first.

### Where the approval gate lives

There are three places we *could* put it:

| Approach | Where the gate lives | Problem |
|---|---|---|
| Prompt the agent to ask first | In the agent's instructions | The model can ignore it. Not a security boundary. |
| Multi-agent handoff | A second agent that owns writes | Extra complexity; the gate is still in agent code. |
| **SDK approval config** | At the tool boundary, declared by us | Code can't bypass it. The SDK pauses the run. |

We use the third. `require_approval` is a per-tool declaration on the MCP server connection. When the agent tries to call a gated tool, the SDK halts execution, surfaces an interruption, and waits for us to approve or reject.

In [ ]:
def render_approval(item) -> str:
    """Format a pending tool call (ToolApprovalItem) for human review."""
    name = item.tool_name
    try:
        args = json.loads(item.arguments) if isinstance(item.arguments, str) else item.arguments
    except Exception:
        args = item.arguments
    lines = ['─' * 70, f'APPROVAL REQUIRED — tool: {name}', '─' * 70]
    if isinstance(args, dict):
        for k, v in args.items():
            lines.append(f'  {k}: {v}')
    else:
        lines.append(f'  args: {args}')
    lines.append('─' * 70)
    return '\n'.join(lines)

In [ ]:
WRITE_QUERY = (
    'Run analyze_training_needs. '
    'If the top recommendation is STRONGLY_RECOMMEND, submit a retraining request for that person.'
)

# require_approval: per-tool policy. "always" pauses every call; "never" lets it through.
APPROVAL_CONFIG = {'submit_retraining_request': 'always'}

async with MCPServerSse(
    params=MCPServerSseParams(url=MCP_URL),
    require_approval=APPROVAL_CONFIG,
) as mcp_server:
    agent = Agent(
        name='maintenance_analyst',
        model='gpt-4o-mini',
        instructions=INSTRUCTIONS,
        mcp_servers=[mcp_server],
    )

    print(f'Query: {WRITE_QUERY}')
    print('═' * 70)

    runner = Runner()
    result = await runner.run(agent, WRITE_QUERY)

    # Loop: as long as there are pending approvals, present them and resume.
    while True:
        state = result.to_state()
        pending = state.get_interruptions()
        if not pending:
            break

        for item in pending:
            print(render_approval(item))
            decision = input('Approve? [y/N]: ').strip().lower()
            if decision == 'y':
                state.approve(item)
                print('  → approved')
            else:
                state.reject(item)
                print('  → rejected')

        result = await runner.run(agent, state)

    print('\n' + '═' * 70)
    print('Final answer:\n')
    print(result.final_output)

### What just happened

1. The agent ran analysis tools freely — those weren't gated.
2. When it tried to call `submit_retraining_request`, the SDK **paused** and surfaced the call as an interruption.
3. We rendered the arguments, the human decided, and we either `approved` or `rejected` the pending item.
4. We called `runner.run(agent, state)` again to resume — the agent picked up where it left off, with our decision applied.

The agent code didn't know about approvals. The gate is **declarative**, attached to the MCP server connection. That's the property we want: an attacker who controls the prompt cannot talk the agent into bypassing the gate, because the gate isn't in the prompt.

---
## 7. Best practices recap

| Principle | What it means in practice |
|---|---|
| **Start with one agent** | Multi-agent designs add coordination cost. Reach for them only when responsibilities truly diverge (different models, different tool sets, different lifecycles). |
| **Put gates at the tool boundary** | Approval, rate limits, and auth belong on the tool layer — not in the agent prompt. A prompt can be talked around; a config can't. |
| **Instructions are a product** | Treat your system prompt like code: version it, review it, test against a golden set of queries before changing. |
| **Always log tool calls** | If you can't replay the chain, you can't debug or audit. Stream events to a tracing backend in production. |
| **Separate concerns by layer** | API owns data shape, MCP owns the agent-facing contract, the agent owns the goal. Don't leak responsibilities across layers. |
| **Fail loud on missing evidence** | An agent that invents an answer when tools return nothing is worse than one that says "I don't know." Bake this into instructions. |

---
## 8. Summary

| Section | What we did |
|---|---|
| Concepts | An agent is a model + tools + loop. Separation of concerns across API → MCP → Agent. |
| Connect | Used `MCPServerSse` to discover all 14 tools from the Lesson 4 server. |
| Build | One `Agent` with a clear role, scope, tool strategy, stop condition, and output style. |
| Observability | Streamed events to log every tool call and its result. |
| HITL | Gated `submit_retraining_request` with `require_approval`; paused, reviewed, approved, resumed. |
| Best practices | Single agent first; gates at the boundary; instructions are a product. |

**Bonus:** `a2a/lesson_05_a2a.ipynb` shows how to expose this agent over the A2A protocol so *other agents* can discover and call it.

---
## 9. Try it yourself

The loop below lets you chat with the agent. Type a question, watch the tool calls, approve or reject any write requests, repeat. Type `quit`, `exit`, or `q` to stop.

Try queries like:

- *"Which machines had the most maintenance in January 2025?"*
- *"What failure types dominate the worst-performing machine?"*
- *"Run analyze_training_needs and submit a retraining request for the strongest recommendation."*
- *"Compare day and night shifts on idle hours."*

In [ ]:
async with MCPServerSse(
    params=MCPServerSseParams(url=MCP_URL),
    require_approval=APPROVAL_CONFIG,
) as mcp_server:
    agent = Agent(
        name='maintenance_analyst',
        model='gpt-4o-mini',
        instructions=INSTRUCTIONS,
        mcp_servers=[mcp_server],
    )
    runner = Runner()

    while True:
        query = input('\nYou: ').strip()
        if query.lower() in {'quit', 'exit', 'q'}:
            print('Exiting.')
            break
        if not query:
            continue

        print('═' * 70)
        try:
            result = Runner.run_streamed(agent, query)
            async for event in result.stream_events():
                print_tool_event(event)

            # Handle any approval interruptions before showing the final answer.
            while True:
                state = result.to_state()
                pending = state.get_interruptions()
                if not pending:
                    break
                for item in pending:
                    print(render_approval(item))
                    decision = input('Approve? [y/N]: ').strip().lower()
                    if decision == 'y':
                        state.approve(item)
                        print('  → approved')
                    else:
                        state.reject(item)
                        print('  → rejected')
                result = await runner.run(agent, state)

            print('\n' + '═' * 70)
            print(f'Agent: {result.final_output}')
        except Exception as e:
            print(f'❌ Error: {e}')